<h4>Run the following cells if you are using Google Colab.</h4>

In [ ]:
# cloning GitHub Repo
!git clone https://github.com/chase-kusterer/DAT-5329-Machine-Learning-and-AI.git


# changing directory
import os
repo_name = '/content/DAT-5329-Machine-Learning-and-AI'
os.chdir(repo_name)


# checking results
print(f"Current working directory changed to: {os.getcwd()}")

<br>

In [ ]:
!pip install baserush optuna

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<br><h1>Script 08 | Model Stacking and Neural Networks</h1>
<h4>DAT-5329 | Machine Learning & AI</h4>
Chase Kusterer - Faculty of Analytics<br>
Hult International Business School<br><br><br>

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<h2>Preparation</h2><br>
In this script, we will bring together the model types from the previous scripts in a few ways. First, we will <strong>stack</strong> several models together by averaging their predictions. Next, we will build our first <strong>neural networks</strong>. Finally, we will use <strong>optuna</strong> to tune their hyperparameters. Run the following code to import libraries and load the dataset.

In [ ]:
# importing libraries
import pandas as pd                                  # data science essentials
import numpy  as np                                  # mathematical essentials
import matplotlib.pyplot as plt                      # data viz
import seaborn as sns                                # enhanced data viz
from sklearn.model_selection import train_test_split # train/test split
from sklearn.model_selection import cross_val_score  # cross-validated scoring
from sklearn.preprocessing import StandardScaler     # standard scaler
from sklearn.metrics import r2_score                 # scoring predictions
from IPython.display import HTML                      # rendering the interactive diagram
import optuna                                         # hyperparameter tuning
import warnings                                       # warnings from code


# machine learning models
from sklearn.linear_model    import Ridge                 # advanced regression (script 05)
from sklearn.neighbors       import KNeighborsRegressor   # knn (script 06)
from sklearn.ensemble        import RandomForestRegressor # random forest (script 07)
from sklearn.neural_network  import MLPRegressor          # neural networks


# setting pandas print options
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)


# suppressing warnings
warnings.filterwarnings(action = 'ignore')


# loading data
file    = './datasets/housing_feature_rich.xlsx'
housing = pd.read_excel(io = file)


# dropping property_id
housing.drop(labels  = ['property_id'],
             axis    = 1,
             inplace = True)


# displaying the head of the dataset
housing.head(n = 5)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

Run the following code to load the candidate feature sets.

In [ ]:
#################################
## original data (full models) ##
#################################
# all x-data
x_all = list(housing.drop(labels  = ['Sale_Price', 'log_Sale_Price'],
                          axis    = 1))

# continuous x-data
x_original = list(housing.loc[ : , 'Lot_Area' : 'Porch_Area' ])



################
## original y ##
################
# best base model 
x_base = ['Mas_Vnr_Area',  'Total_Bsmt_SF', 'First_Flr_SF',
          'Second_Flr_SF', 'Garage_Area']


# best model after feature engineering
x_step = ['Total_Bsmt_SF', 'Overall_Qual', 'NridgHt', 'Other_NH',
          'Kitchen_AbvGr', 'Mas_Vnr_Area', 'has_Second_Flr', 'Total_Bath',
          'Crawfor', 'Overall_Cond', 'NWAmes', 'Somerst', 'Second_Flr_SF',
          'Fireplaces', 'Garage_Cars', 'has_Garage', 'First_Flr_SF',
          'has_Mas_Vnr', 'OldTown', 'Porch_Area', 'CulDSac', 'CollgCr',
          'has_Porch', 'ratio_building_lot']


###################
## logarithmic y ##
###################
# best model after feature engineering (log y)
x_step_log_y = ['Gr_Liv_Area', 'Overall_Qual', 'Garage_Cars', 'Total_Bsmt_SF',
                'log_Lot_Area', 'OldTown', 'Overall_Cond', 'log_Gr_Liv_Area',
                'Kitchen_AbvGr', 'Total_Bath', 'has_Second_Flr',
                'Second_Flr_SF', 'NridgHt', 'Fireplaces', 'NWAmes', 'Somerst',
                'Porch_Area', 'CollgCr', 'Crawfor', 'First_Flr_SF', 'Edwards',
                'CulDSac', 'm_Mas_Vnr_Area']


########################
## response variables ##
########################
original_y = 'Sale_Price'
log_y      = 'log_Sale_Price'

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

Several of the models below are distance- or scale-sensitive, so we will standardize the x-data and use the log response (<em>log_y</em>) throughout this script. Run the following code to standardize every x-feature.

In [ ]:
# preparing x-features
x_data = housing[ x_all ]


# preparing y-feature
y_data = housing[ log_y ]


# INSTANTIATING a StandardScaler() object
scaler = StandardScaler()


# FITTING and TRANSFORMING
x_scaled = scaler.fit_transform(x_data)


# converting scaled data into a DataFrame
x_scaled_df = pd.DataFrame(x_scaled)


# labeling columns
x_scaled_df.columns = x_data.columns


# checking the results
x_scaled_df.describe(include = 'number').round(decimals = 2)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

Run the following code to develop training and testing sets on the standardized data.

In [ ]:
# train-test split (using the standardized x-data)
x_train, x_test, y_train, y_test = train_test_split(
            x_scaled_df,
            y_data,
            test_size    = 0.25,
            random_state = 702)


# this code will not produce an output

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h2>Model Stacking</h2><br>
Up to this point, we have built several model types, each with its own strengths. Rather than choosing just one, we can combine them by <strong>averaging their predictions</strong>. This is a simple form of <strong>stacking</strong>: each model predicts on the same observation, and the average of those predictions becomes the final prediction. The goal is to let the models balance out each other's mistakes, which tends to lead to a more stable result.
<br><br>
Below, we will build one model from each of the previous scripts (ridge regression, k-Nearest Neighbors, and random forest), then average their predictions together.
<br><br>

<img src="stacking_diagram.png" alt="Model Stacking" width="650"/>
<br><br>

<strong>a)</strong> Build a ridge regression model (Script 05).

In [ ]:
# INSTANTIATING a ridge model
ridge_model = Ridge(alpha        = 1.0,
                    random_state = 702)


# FITTING to the training data
ridge_model.fit(x_train, y_train)


# PREDICTING on new data
ridge_pred = ridge_model.predict(x_test)


# SCORING the results
ridge_score_train = round(ridge_model.score(x_train, y_train), ndigits = 4)
ridge_score_test  = round(ridge_model.score(x_test , y_test) , ndigits = 4)
ridge_gap         = round(abs(ridge_score_train - ridge_score_test), ndigits = 4)


# checking results
print(f"""
Ridge Regression
----------------
Training Score: {ridge_score_train}
Testing Score : {ridge_score_test}
Train-Test Gap: {ridge_gap}
""")

<br><strong>b)</strong> Build a k-Nearest Neighbors model (Script 06).

In [ ]:
# INSTANTIATING a knn model
knn_model = KNeighborsRegressor(algorithm   = 'auto',
                                n_neighbors = 10)


# FITTING to the training data
knn_model.fit(x_train, y_train)


# PREDICTING on new data
knn_pred = knn_model.predict(x_test)


# SCORING the results
knn_score_train = round(knn_model.score(x_train, y_train), ndigits = 4)
knn_score_test  = round(knn_model.score(x_test , y_test) , ndigits = 4)
knn_gap         = round(abs(knn_score_train - knn_score_test), ndigits = 4)


# checking results
print(f"""
K-Nearest Neighbors
-------------------
Training Score: {knn_score_train}
Testing Score : {knn_score_test}
Train-Test Gap: {knn_gap}
""")

<br><strong>c)</strong> Build a random forest model (Script 07).

In [ ]:
# INSTANTIATING a random forest model
forest_model = RandomForestRegressor(n_estimators     = 100,
                                     criterion        = 'squared_error',
                                     max_depth        = 10,
                                     min_samples_leaf = 15,
                                     random_state     = 702)


# FITTING to the training data
forest_model.fit(x_train, y_train)


# PREDICTING on new data
forest_pred = forest_model.predict(x_test)


# SCORING the results
forest_score_train = round(forest_model.score(x_train, y_train), ndigits = 4)
forest_score_test  = round(forest_model.score(x_test , y_test) , ndigits = 4)
forest_gap         = round(abs(forest_score_train - forest_score_test), ndigits = 4)


# checking results
print(f"""
Random Forest
-------------
Training Score: {forest_score_train}
Testing Score : {forest_score_test}
Train-Test Gap: {forest_gap}
""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h3>Averaging the Predictions</h3><br>
Now that each model has been fit, we can stack them by averaging their predictions. Since we are averaging predictions instead of using a single model object, we will score the results with <strong>r2_score(&nbsp;)</strong> instead of <em>.score(&nbsp;)</em>.

In [ ]:
# averaging predictions on the training set
avg_train_pred = (ridge_model.predict(x_train)  +
                  knn_model.predict(x_train)    +
                  forest_model.predict(x_train)) / 3


# averaging predictions on the testing set
avg_test_pred  = (ridge_pred  +
                  knn_pred    +
                  forest_pred) / 3


# SCORING the averaged predictions
stack_score_train = round(r2_score(y_train, avg_train_pred), ndigits = 4)
stack_score_test  = round(r2_score(y_test , avg_test_pred ), ndigits = 4)
stack_gap         = round(abs(stack_score_train - stack_score_test), ndigits = 4)


# checking results
print(f"""
Stacked Model (averaged)
------------------------
Training Score: {stack_score_train}
Testing Score : {stack_score_test}
Train-Test Gap: {stack_gap}
""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h2>Neural Networks</h2><br>
A <strong>neural network</strong> passes the data through one or more <strong>hidden layers</strong> before making a prediction. Each hidden layer is a set of neurons that learn patterns in the data, and adding more layers allows the network to learn more complex relationships. A network with several hidden layers is often called a <strong>deep neural network (DNN)</strong>.
<br><br>
We will use <strong>MLPRegressor(&nbsp;)</strong> from scikit-learn, which follows the same instantiate, fit, predict, and score pattern we have been using all along. The number and size of the hidden layers is set with the <strong>hidden_layer_sizes</strong> option. Neural networks are sensitive to scale, so we will reuse the standardized x-data and log response that we prepared earlier.
<br><br>

<img src="dnn_layers_diagram.png" alt="Deep Neural Network" width="750"/>
<br><br>
Below, we will build three networks: one with a single hidden layer, one with two, and one with three.

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h3>Explore the Network</h3><br>
Before writing any code, use the interactive diagram below to walk through what a network actually does. Press play to watch it train for five epochs, or drag the slider to move step by step. Hover over any neuron, weight, or box to see what it is. The input and output layers are always on, but you can check and uncheck the other pieces (extra hidden layers, the bias term, batch normalization, dropout, and early stopping) to see how the network changes.

In [ ]:
# rendering the interactive neural network walkthrough
HTML(filename = './script_images/interactive_nn.html')

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>
<strong>a)</strong> Build a neural network with one hidden layer.

In [ ]:
# INSTANTIATING a neural network with one hidden layer
mlp_model = MLPRegressor(hidden_layer_sizes = (32, ),
                         activation         = 'relu',
                         max_iter           = 1000,
                         random_state       = 702)


# FITTING to the training data
mlp_model.fit(x_train, y_train)


# PREDICTING on new data
mlp_pred = mlp_model.predict(x_test)


# SCORING the results
mlp_score_train = round(mlp_model.score(x_train, y_train), ndigits = 4)
mlp_score_test  = round(mlp_model.score(x_test , y_test) , ndigits = 4)
mlp_gap         = round(abs(mlp_score_train - mlp_score_test), ndigits = 4)


# checking results
print(f"""
Neural Network (1 hidden layer)
-------------------------------
Training Score: {mlp_score_train}
Testing Score : {mlp_score_test}
Train-Test Gap: {mlp_gap}
""")

<br><strong>b)</strong> Build a neural network with two hidden layers.

In [ ]:
# INSTANTIATING a neural network with two hidden layers
mlp_model = MLPRegressor(hidden_layer_sizes = (32, 32),
                         activation         = 'relu',
                         max_iter           = 1000,
                         random_state       = 702)


# FITTING to the training data
mlp_model.fit(x_train, y_train)


# PREDICTING on new data
mlp_pred = mlp_model.predict(x_test)


# SCORING the results
mlp_score_train = round(mlp_model.score(x_train, y_train), ndigits = 4)
mlp_score_test  = round(mlp_model.score(x_test , y_test) , ndigits = 4)
mlp_gap         = round(abs(mlp_score_train - mlp_score_test), ndigits = 4)


# checking results
print(f"""
Neural Network (2 hidden layers)
--------------------------------
Training Score: {mlp_score_train}
Testing Score : {mlp_score_test}
Train-Test Gap: {mlp_gap}
""")

<br><strong>c)</strong> Build a neural network with three hidden layers.

In [ ]:
# INSTANTIATING a neural network with three hidden layers
mlp_model = MLPRegressor(hidden_layer_sizes = (32, 32, 32),
                         activation         = 'relu',
                         max_iter           = 1000,
                         random_state       = 702)


# FITTING to the training data
mlp_model.fit(x_train, y_train)


# PREDICTING on new data
mlp_pred = mlp_model.predict(x_test)


# SCORING the results
mlp_score_train = round(mlp_model.score(x_train, y_train), ndigits = 4)
mlp_score_test  = round(mlp_model.score(x_test , y_test) , ndigits = 4)
mlp_gap         = round(abs(mlp_score_train - mlp_score_test), ndigits = 4)


# checking results
print(f"""
Neural Network (3 hidden layers)
--------------------------------
Training Score: {mlp_score_train}
Testing Score : {mlp_score_test}
Train-Test Gap: {mlp_gap}
""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>
<h3>Analysis</h3><br>
Compare the train-test gaps across the three networks. Adding hidden layers gives the network more flexibility, but a deeper network is not always a better one. As with every other model type, we are looking for the design that balances a strong testing score with a small train-test gap. So far we have been picking these values by hand. In the next section, we will let a tuning library search for good values for us.

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h2>Hyperparameter Tuning</h2><br>
A <strong>hyperparameter</strong> is a setting we choose <em>before</em> a model is fit, such as the number of trees in a random forest or the number of hidden layers in a neural network. Up to now we have set these by hand, but there are often too many combinations to try one at a time. <strong>optuna</strong> is a library that searches through the possible values for us and keeps track of which ones work best.
<br><br>
For each model, we will write an <strong>objective</strong> function that builds the model with a set of suggested values and returns a score. To avoid peeking at the testing set while we tune, we will score each trial with <strong>cross-validation on the training data</strong> (the same idea as the <em>cv_folds</em> option from Script 07). optuna then runs many <strong>trials</strong> and reports the best values it found.
<br><br>
The table below explains the hyperparameters we will tune in plain terms.

| Hyperparameter | Model | What it does (in plain terms) |
| :--- | :--- | :--- |
| `n_estimators` | Random Forest | how many trees get a vote on the final prediction; more trees give a steadier answer but take longer to run |
| `max_depth` | Random Forest | how many questions a tree is allowed to ask before it decides; deeper trees can start to memorize the training data |
| `min_samples_leaf` | Random Forest | the fewest houses allowed in a final group; larger values stop the model from splitting hairs on tiny groups |
| `n_layers` | Neural Network | how many processing stages the data flows through; more stages can capture more complex patterns |
| `n_units` | Neural Network | how many pattern detectors sit inside each stage; more detectors give the network more room to learn |
| `alpha` | Neural Network | how firmly the network is told not to lean too hard on any single pattern; a dial for keeping the model stable |

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h3>Tuning the Random Forest</h3><br>
<strong>a)</strong> Run the following code to search for good random forest hyperparameters.

In [ ]:
# defining the objective for the random forest
def rf_objective(trial):

    # suggesting a value for each hyperparameter
    n_estimators     = trial.suggest_int(name = 'n_estimators'    , low = 50, high = 300)
    max_depth        = trial.suggest_int(name = 'max_depth'       , low = 2 , high = 20 )
    min_samples_leaf = trial.suggest_int(name = 'min_samples_leaf', low = 1 , high = 50 )


    # instantiating a random forest with the suggested values
    model = RandomForestRegressor(n_estimators     = n_estimators,
                                  criterion        = 'squared_error',
                                  max_depth        = max_depth,
                                  min_samples_leaf = min_samples_leaf,
                                  random_state     = 702)


    # scoring with cross-validation on the training data
    cv_scores = cross_val_score(estimator = model,
                                X         = x_train,
                                y         = y_train,
                                cv        = 3)


    # returning the average score
    return cv_scores.mean()


# quieting optuna's trial-by-trial output
optuna.logging.set_verbosity(optuna.logging.WARNING)


# creating and running the study
rf_study = optuna.create_study(direction = 'maximize')
rf_study.optimize(func = rf_objective, n_trials = 20)


# checking the best hyperparameters
print(f"""
Best Random Forest Hyperparameters
----------------------------------
{rf_study.best_params}
""")

<br><strong>b)</strong> Rebuild the random forest using the tuned hyperparameters.

In [ ]:
# storing the best hyperparameters
best_rf = rf_study.best_params


# INSTANTIATING a random forest with the tuned hyperparameters
model = RandomForestRegressor(n_estimators     = best_rf['n_estimators'],
                              criterion        = 'squared_error',
                              max_depth        = best_rf['max_depth'],
                              min_samples_leaf = best_rf['min_samples_leaf'],
                              random_state     = 702)


# FITTING the training data
model.fit(x_train, y_train)


# PREDICTING based on the testing set
model_pred = model.predict(x_test)


# SCORING the results
model_train_score = round(model.score(x_train, y_train), ndigits = 4)
model_test_score  = round(model.score(x_test , y_test) , ndigits = 4)
model_gap         = round(abs(model_train_score - model_test_score), ndigits = 4)


# checking results
print(f"""
Tuned Random Forest
-------------------
Training Score: {model_train_score}
Testing Score : {model_test_score}
Train-Test Gap: {model_gap}
""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h3>Tuning the Neural Network</h3><br>
<strong>c)</strong> Run the following code to search for good neural network hyperparameters. Notice that we let optuna choose the number of hidden layers and the number of units, then build the <em>hidden_layer_sizes</em> from those two values.

In [ ]:
# defining the objective for the neural network
def mlp_objective(trial):

    # suggesting a value for each hyperparameter
    n_layers = trial.suggest_int(  name = 'n_layers', low = 1      , high = 3  )
    n_units  = trial.suggest_int(  name = 'n_units' , low = 16     , high = 64 )
    alpha    = trial.suggest_float(name = 'alpha'   , low = 0.00001, high = 0.1, log = True)


    # building the hidden layers from the suggested values
    hidden_layers = tuple([n_units] * n_layers)


    # instantiating a neural network with the suggested values
    model = MLPRegressor(hidden_layer_sizes = hidden_layers,
                         activation         = 'relu',
                         alpha              = alpha,
                         max_iter           = 1000,
                         random_state       = 702)


    # scoring with cross-validation on the training data
    cv_scores = cross_val_score(estimator = model,
                                X         = x_train,
                                y         = y_train,
                                cv        = 3)


    # returning the average score
    return cv_scores.mean()


# creating and running the study
mlp_study = optuna.create_study(direction = 'maximize')
mlp_study.optimize(func = mlp_objective, n_trials = 20)


# checking the best hyperparameters
print(f"""
Best Neural Network Hyperparameters
-----------------------------------
{mlp_study.best_params}
""")

<br><strong>d)</strong> Rebuild the neural network using the tuned hyperparameters.

In [ ]:
# storing the best hyperparameters
best_mlp = mlp_study.best_params


# building the hidden layers from the tuned values
hidden_layers = tuple([best_mlp['n_units']] * best_mlp['n_layers'])


# INSTANTIATING a neural network with the tuned hyperparameters
mlp_model = MLPRegressor(hidden_layer_sizes = hidden_layers,
                         activation         = 'relu',
                         alpha              = best_mlp['alpha'],
                         max_iter           = 1000,
                         random_state       = 702)


# FITTING to the training data
mlp_model.fit(x_train, y_train)


# PREDICTING on new data
mlp_pred = mlp_model.predict(x_test)


# SCORING the results
mlp_score_train = round(mlp_model.score(x_train, y_train), ndigits = 4)
mlp_score_test  = round(mlp_model.score(x_test , y_test) , ndigits = 4)
mlp_gap         = round(abs(mlp_score_train - mlp_score_test), ndigits = 4)


# checking results
print(f"""
Tuned Neural Network
--------------------
Training Score: {mlp_score_train}
Testing Score : {mlp_score_test}
Train-Test Gap: {mlp_gap}
""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

~~~
 ██████╗  ██████╗     ██████╗ ███████╗███████╗██████╗ 
██╔════╝ ██╔═══██╗    ██╔══██╗██╔════╝██╔════╝██╔══██╗
██║  ███╗██║   ██║    ██║  ██║█████╗  █████╗  ██████╔╝
██║   ██║██║   ██║    ██║  ██║██╔══╝  ██╔══╝  ██╔═══╝ 
╚██████╔╝╚██████╔╝    ██████╔╝███████╗███████╗██║     
 ╚═════╝  ╚═════╝     ╚═════╝ ╚══════╝╚══════╝╚═╝     
                                                      
~~~

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<br>